This notebook is my practice of using tools in Claude API.

In this practice, I will first ask Claude a question about how old are my kids. Without any knowledge, Claude shouldn't be able to answer this question.
Then I will provide two tools, one tool can return my kids' names and the other tool takes a name and return the age.
With the help of these two tools, when ask the question again, Claude should be able to return the correct answers.

In [1]:
# Load env variables and create client
from dotenv import load_dotenv
from anthropic import Anthropic
from datetime import datetime, timedelta
import json

load_dotenv()

client = Anthropic()
model = "claude-haiku-4-5"

In [2]:
from anthropic.types import ToolParam

def get_all_kids():
    return ["Alice", "Bob", "Charlie"]

# Use Claude chat online to get the best schema for this function

get_all_kids_schema = ToolParam(
    {
      "name": "get_all_kids",
      "description": "Return the full list of kids' names.",
      "input_schema": 
      {
        "type": "object",
        "properties": {},
        "required": []
      }
    }
)

In [3]:
def get_kids_age(name: str):
    ages = {
        "Alice": 7,
        "Bob": 9,
        "Charlie": 6
    }
    return ages.get(name, None)

get_kids_age_schema = ToolParam(
    {
        "name": "get_kids_age",
        "description": "Get the age of a specific kid by name. Returns None if the name is not found.",
        "input_schema": 
        {
            "type": "object",
            "properties": 
            {
                "name": 
                {
                    "type": "string",
                    "description": "The name of the kid (e.g., 'Alice', 'Bob', 'Charlie')"
                }
            },
            "required": ["name"]
        }
    }
)

In [4]:
# Helper functions
from anthropic.types import Message

def add_user_message(messages, message):
    user_message = {
        "role": "user",
        "content": message.content if isinstance(message, Message) else message,
    }
    messages.append(user_message)


def add_assistant_message(messages, message):
    assistant_message = {
        "role": "assistant",
        "content": message.content if isinstance(message, Message) else message,
    }
    messages.append(assistant_message)


def chat(messages, system=None, temperature=1.0, stop_sequences=[], tools=None):
    params = {
        "model": model,
        "max_tokens": 1000,
        "messages": messages,
        "temperature": temperature,
        "stop_sequences": stop_sequences,
    }

    if system:
        params["system"] = system

    if tools:
        params["tools"] = tools

    message = client.messages.create(**params)
    return message

def text_from_message(message):
    return "\n".join([block.text for block in message.content if block.type == "text"])

In [5]:
messages = []
add_user_message(messages, "What are the ages of my kids?")
response = chat(messages=messages)
print(text_from_message(response))

I don't have any information about you or your children. I don't have access to personal information about users unless you share it with me in our conversation.

If you'd like to tell me about your kids' ages, I'm happy to listen!


In [6]:
def run_conversation(messages):
    while True:
        response = chat(messages, tools=[get_all_kids_schema, get_kids_age_schema])

        add_assistant_message(messages, response)
        # print(text_from_message(response))

        if response.stop_reason != "tool_use":
            break

        tool_result_blocks = run_tools(response)
        add_user_message(messages, tool_result_blocks)

    return messages

def run_tools(response):
    tool_requests = [block for block in response.content if block.type == "tool_use"]
    tool_result_blocks = []

    for tool_request in tool_requests:
        try:
            tool_output = run_tool(tool_request.name, tool_request.input)
            tool_result_block = {
                "type": "tool_result",
                "tool_use_id": tool_request.id,
                "content": json.dumps(tool_output),
                "is_error": False,
            }
        except Exception as e:
            tool_result_block = {
                "type": "tool_result",
                "tool_use_id": tool_request.id,
                "content": f"Error: {e}",
                "is_error": True,
            }

        tool_result_blocks.append(tool_result_block)

    return tool_result_blocks

def run_tool(name, input):
    if name == "get_all_kids":
        return get_all_kids()
    elif name == "get_kids_age":
        return get_kids_age(input["name"])
    else:
        raise ValueError(f"Unknown tool: {name}")
    
messages = []
add_user_message(messages, "What are the ages of my kids?")
run_conversation(messages)
print(messages[-1]["content"][0].text)

Here are the ages of your kids:

- **Alice**: 7 years old
- **Bob**: 9 years old
- **Charlie**: 6 years old


In [66]:
print(messages[-1]["content"][0].text)

Here are the ages of your kids:

- **Alice**: 7 years old
- **Bob**: 9 years old
- **Charlie**: 6 years old


In [9]:
get_current_datetime_schema = ToolParam({
  "name": "get_current_datetime",
  "description": "Get the current date and time formatted according to the specified format string. Uses Python's strftime format codes (e.g., %Y for year, %m for month, %d for day, %H for hour, %M for minute, %S for second).",
  "input_schema": {
    "type": "object",
    "properties": {
      "date_format": {
        "type": "string",
        "description": "The format string for the datetime output using strftime format codes. Default is '%Y-%m-%d %H:%M:%S' (e.g., '2024-12-04 14:30:45'). Common format codes include: %Y (4-digit year), %m (month 01-12), %d (day 01-31), %H (hour 00-23), %M (minute 00-59), %S (second 00-59), %A (weekday name), %B (month name).",
        "default": "%Y-%m-%d %H:%M:%S"
      }
    },
    "required": []
  }
})

In [6]:
# Helper functions
def add_user_message(messages, text):
    user_message = {"role": "user", "content": text}
    messages.append(user_message)


def add_assistant_message(messages, text):
    assistant_message = {"role": "assistant", "content": text}
    messages.append(assistant_message)


def chat(messages, system=None, temperature=1.0, stop_sequences=[], tools=None):
    params = {
        "model": model,
        "max_tokens": 1000,
        "messages": messages,
        "temperature": temperature,
        "stop_sequences": stop_sequences,
    }

    if system:
        params["system"] = system

    if tools:
        params["tools"] = tools

    message = client.messages.create(**params)
    return message

def text_from_message(message):
    return "\n".join([block.text for block in message.content if block.type == "text"])



In [ ]:
def run_conversation(messages):
    while True:
        response = chat(messages, tools=[get_current_datetime_schema])

        add_assistant_message(messages, response)
        print(text_from_message(response))

        add_user_message(messages, response)

        if response isn't asking for a tool:
            break

        tool_result_blocks = run_tools(response)
        add_user_message(tool_result_blocks)

    return messages

In [ ]:
def run_tools(message):
    

In [13]:
messages = []

add_user_message(messages, "Hello, what is the current date and time?")

In [14]:
print(messages)

[{'role': 'user', 'content': 'Hello, what is the current date and time?'}]


In [15]:
response = chat(messages)
print(response)

Message(id='msg_01DJ6HHcVmNyqAthx6feZ9kY', content=[TextBlock(citations=None, text='I don\'t have access to real-time information, so I can\'t tell you the current date and time. \n\nTo find out what time it is right now, you can:\n- Check your device (phone, computer, watch)\n- Search "current time" online\n- Ask a voice assistant like Siri, Alexa, or Google Assistant\n\nIs there something else I can help you with?', type='text')], model='claude-haiku-4-5-20251001', role='assistant', stop_reason='end_turn', stop_sequence=None, type='message', usage=Usage(cache_creation=CacheCreation(ephemeral_1h_input_tokens=0, ephemeral_5m_input_tokens=0), cache_creation_input_tokens=0, cache_read_input_tokens=0, input_tokens=17, output_tokens=90, server_tool_use=None, service_tier='standard'))


In [ ]:
messages.append(response)
print(messages)

In [41]:
messages = []

messages.append(
    {
        "role": "user",
        "content": "What is the exact time, formatted as HH:MM:SS?"
    }
)

response = client.messages.create(
    model=model,
    max_tokens=1000,
    messages=messages,
    tools=[get_current_datetime_schema]
)

In [42]:
response

Message(id='msg_01JT5YXEfHczpDypLRaJq6ov', content=[ToolUseBlock(id='toolu_01KTZuxxszoRLGSAyhRBpPK4', input={'date_format': '%H:%M:%S'}, name='get_current_datetime', type='tool_use')], model='claude-haiku-4-5-20251001', role='assistant', stop_reason='tool_use', stop_sequence=None, type='message', usage=Usage(cache_creation=CacheCreation(ephemeral_1h_input_tokens=0, ephemeral_5m_input_tokens=0), cache_creation_input_tokens=0, cache_read_input_tokens=0, input_tokens=764, output_tokens=63, server_tool_use=None, service_tier='standard'))

In [43]:
messages.append({
    "role": "assistant",
    "content": response.content
})

messages

[{'role': 'user', 'content': 'What is the exact time, formatted as HH:MM:SS?'},
 {'role': 'assistant',
  'content': [ToolUseBlock(id='toolu_01KTZuxxszoRLGSAyhRBpPK4', input={'date_format': '%H:%M:%S'}, name='get_current_datetime', type='tool_use')]}]

In [45]:
result = get_current_datetime(**response.content[0].input)

In [46]:
result

'22:43:33'

In [47]:
messages.append({
    "role": "user",
    "content": [
        {
            "type": "tool_result",
            "tool_use_id": response.content[0].id,
            "content": result
        }
    ]
})

In [48]:
messages

[{'role': 'user', 'content': 'What is the exact time, formatted as HH:MM:SS?'},
 {'role': 'assistant',
  'content': [ToolUseBlock(id='toolu_01KTZuxxszoRLGSAyhRBpPK4', input={'date_format': '%H:%M:%S'}, name='get_current_datetime', type='tool_use')]},
 {'role': 'user',
  'content': [{'type': 'tool_result',
    'tool_use_id': 'toolu_01KTZuxxszoRLGSAyhRBpPK4',
    'content': '22:43:33'}]}]

In [49]:
response = client.messages.create(
    model=model,
    max_tokens=1000,
    messages=messages,
    tools=[get_current_datetime_schema]
)

In [50]:
response

Message(id='msg_01ACKcA8amdwwrNRGkGsZ45U', content=[TextBlock(citations=None, text='The exact time is **22:43:33** (in HH:MM:SS format).', type='text')], model='claude-haiku-4-5-20251001', role='assistant', stop_reason='end_turn', stop_sequence=None, type='message', usage=Usage(cache_creation=CacheCreation(ephemeral_1h_input_tokens=0, ephemeral_5m_input_tokens=0), cache_creation_input_tokens=0, cache_read_input_tokens=0, input_tokens=844, output_tokens=24, server_tool_use=None, service_tier='standard'))